In [45]:
import numpy as np
import pandas as pd
import os
import h5py
import ast


RESULTS_FOLDER_PATH = "data/processing/batches"
PROPERTIES_PATH = "data/target_properties_v2.csv"
SPECTRA_PATH = "data/spectra.h5"
RESULTS_PATH = "data/results.csv"
FIT_PARAMS_PATH = "data/fit_params.csv"
DETAILS_PATH = "data/result_details.csv"
RELEVANT_RESULTS_PATH = "data/relevant_results.csv"

In [ ]:
# # files_list = os.listdir(RESULTS_FOLDER_PATH)

# data = pd.read_json(f"{RESULTS_FOLDER_PATH}/{files_list[0]}",lines=True,dtype={"targetid":"str"})

# data = data[['targetid', 'rew_civ', 'rew_nv', 'rew_lya', 'snr_1700A', 'snr_civ', 'snr_nv', 'snr_lya', 'pixel_used_civ', 'pixel_used_blue_end_lya', 'continuum_redchi']].copy()

# for f in files_list[1:]:
#     temp = pd.read_json(f"{RESULTS_FOLDER_PATH}/{f}",lines=True,dtype={"targetid":"str"})
#     temp = temp[['targetid', 'rew_civ', 'rew_nv', 'rew_lya', 'snr_1700A', 'snr_civ', 'snr_nv', 'snr_lya', 'pixel_used_civ', 'pixel_used_blue_end_lya', 'continuum_redchi']].copy()

#     data = pd.concat([data,temp])

# data.info()

In [1]:
# properties = pd.read_csv(PROPERTIES_PATH,index_col=0,dtype={"targetid":"str"})
# results = data.merge(properties,how="left",on="targetid")

# results.to_csv(RESULTS_PATH)

# results = pd.read_csv(RESULTS_PATH,index_col=0,dtype={"targetid":"str"})

# results.info()

In [ ]:
# results["quality_flag"] = (results["snr_civ"] < 2.5) | (results["continuum_redchi"] > 2) | results["rew_civ"].isna()

# results["flag_civ"] = results["rew_civ"].isna() | (results["pixel_used_civ"] < 0.7) | results["quality_flag"]
# results["flag_nv"] = results["rew_nv"].isna() | results["rew_lya"].isna() | results["quality_flag"]
# results["flag_lya"] = results["rew_lya"].isna() | results["rew_nv"].isna() | (results["pixel_used_blue_end_lya"] < 0.65) | results["quality_flag"]

In [ ]:
# valid = (results["flux_w3"] > 0) & (results["flux_z"] > 0)
# results["z-w3"] = np.full_like(results["flux_w3"], np.nan)
# results["z-w3"] = 2.5 * np.log10(results["flux_w3"][valid] / results["flux_z"][valid])

# # results["snr_w3"] = results["flux_w3"] * np.sqrt(results["flux_ivar_w3"])

In [22]:
# results.to_csv(RESULTS_PATH)

In [2]:
# results[(results["rew_civ"] > 100) & (results["flag_civ"]==False) & (results["z-w3"] > 3.9) & (2 < results["z"]) & (results["z"] < 3.4) & (results["snr_w3"] > 3)].sort_values("rew_civ")

### final version of file to use

In [36]:
results = pd.read_csv(RESULTS_PATH,index_col=0,dtype={"targetid":"str"})

In [ ]:
results["warning"] = (results["snr_civ"] < 2.5) | (results["continuum_redchi"] > 2) | results["rew_civ"].isna() | (results["rew_civ"] < 0) | (results["pixel_used_civ"] < 0.7) | results["rew_nv"].isna() | (results["rew_nv"] < 0) | results["rew_lya"].isna() | (results["rew_lya"] < 0) | (results["pixel_used_blue_end_lya"] < 0.65) | (results["flux_z"] <= 0) | (results["flux_z"].isna()) 

# good result:
# SNR CIV > 2.5
# continuum redchi2 < 2
# all REW > 0 and not null
# CIV at least 70% pixel used
# Lya at least 65% pixel used on left side
# z flux is larger 0 and not null

In [38]:
stats = results[results["warning"]==False].copy()
stats = stats[['targetid', 'rew_civ', 'rew_nv', 'rew_lya', 'snr_1700A', 'snr_civ',
       'snr_nv', 'snr_lya', 'pixel_used_civ', 'pixel_used_blue_end_lya',
       'continuum_redchi', 'z', 'zerr', 'civ_1549_flux', 'civ_1549_flux_ivar',
       'flux_z', 'flux_ivar_z', 'flux_w3', 'flux_ivar_w3', 'desiname', 'z-w3', 'snr_w3',
       'warning']]

In [16]:
details = pd.read_csv(DETAILS_PATH,index_col=0,dtype={"targetid":"str"})
details = details[details["targetid"].isin(stats["targetid"])]

In [ ]:
# extract fit REW values
details["fit_rew_civ"] = details["measurements_civ"].apply(lambda x: ast.literal_eval(str(x))["further_measurements"]["fit_rew"] if x != np.nan else None)
details["fit_rew_nv"] = details["measurements_nv"].apply(lambda x: ast.literal_eval(str(x))["further_measurements"]["fit_rew"] if x != np.nan else None)
details["fit_rew_lya"] = details["measurements_lya"].apply(lambda x: ast.literal_eval(str(x))["further_measurements"]["fit_rew"] if x != np.nan else None)

# extract flux values
details["flux_civ"] = details["measurements_civ"].apply(lambda x: ast.literal_eval(str(x))["further_measurements"]["line_flux"] if x != np.nan else None)
details["flux_nv"] = details["measurements_nv"].apply(lambda x: ast.literal_eval(str(x))["further_measurements"]["line_flux"] if x != np.nan else None)
details["flux_lya"] = details["measurements_lya"].apply(lambda x: ast.literal_eval(str(x))["further_measurements"]["line_flux"] if x != np.nan else None)

# extract fwhm values
details["fwhm_civ"] = details["measurements_civ"].apply(lambda x: ast.literal_eval(str(x))["further_measurements"]["fwhm_kms"] if x != np.nan else None)
details["fwhm_nv"] = details["measurements_nv"].apply(lambda x: ast.literal_eval(str(x))["further_measurements"]["fwhm_kms"] if x != np.nan else None)
details["fwhm_lya"] = details["measurements_lya"].apply(lambda x: ast.literal_eval(str(x))["further_measurements"]["fwhm_kms"] if x != np.nan else None)

# extract kt80 values
details["kt80_civ"] = details["measurements_civ"].apply(lambda x: ast.literal_eval(str(x))["further_measurements"]["fwhm_kms"] if x != np.nan else None)
details["kt80_nv"] = details["measurements_nv"].apply(lambda x: ast.literal_eval(str(x))["further_measurements"]["fwhm_kms"] if x != np.nan else None)
details["kt80_lya"] = details["measurements_lya"].apply(lambda x: ast.literal_eval(str(x))["further_measurements"]["fwhm_kms"] if x != np.nan else None)

details = details[['targetid', 'fit_rew_civ', 'fit_rew_nv', 'fit_rew_lya', 'flux_civ', 'flux_nv', 'flux_lya', 'fwhm_civ', 'fwhm_nv', 'fwhm_lya', 'kt80_civ', 'kt80_nv', 'kt80_lya']]

In [39]:
stats = stats.merge(details,how="left",on="targetid")

In [ ]:
stats = stats.set_index("targetid")

stats.to_csv(RELEVANT_RESULTS_PATH)

In [46]:
test = pd.read_csv(RELEVANT_RESULTS_PATH,index_col=0,dtype={"targetid":"str"})